# DEXTRA · 04 — iTransformer backbone

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aduitya9/DEXTRA/blob/main/notebooks/04_itransformer.ipynb)

The forecasting backbone, and the checkpoint the rest of DEXTRA is built on.

### What makes it *i*Transformer
A conventional time-series transformer makes one token per time step, so attention asks which **days** matter to each other. iTransformer ([Liu et al., ICLR 2024](https://arxiv.org/abs/2310.06625)) inverts that: each variate's whole 14-day series is one token, and attention asks which **variates** matter to each other.

That is the question worth asking here. Notebook 02 measured four features whose relationship with AQI *reverses between seasons* — `blh`, `blh_anomaly`, `fire_count_500km` and `temp2m`. Reading them together is the whole point, and it costs no more attention for a 14-day window than a 3-day one, because a token is a series rather than a day.

### One question only
Training is plain mean squared error. Weighting the loss towards hazardous days would improve the number this project cares about — which is exactly why it is not done here. This notebook has to answer whether the **architecture** earns its place; changing the loss at the same time would answer two questions at once and neither cleanly. Rare-event handling arrives in notebook 06 with C-STDA, where it can be measured on its own.

### The bar to clear
From notebook 03, on 2025:

| | RMSE | rare-event F1 |
|---|---|---|
| persistence | 33.88 | 0.680 |
| gradient boosting | **29.11** | **0.705** |
| *on unseen cities* | 33.82 | 0.537 |

### Runtime
**GPU (T4)** if you have one; it also runs on CPU in a few minutes. Set the runtime before you start.

In [ ]:
# Leave empty to read from the HF_DATASET_REPO Colab secret.
HF_DATASET_REPO = ""
GITHUB_REPO_URL = "https://github.com/Aduitya9/DEXTRA.git"

# Architecture
D_MODEL, N_HEADS, N_LAYERS, DROPOUT = 128, 8, 2, 0.3

# Training
EPOCHS, BATCH_SIZE, LEARNING_RATE, PATIENCE, SEED = 30, 256, 3e-4, 10, 42
WEIGHT_DECAY = 1e-3

SAVE_TO_HF = True   # upload the checkpoint and the forecasts when done

## Secrets
`HF_TOKEN` and `HF_DATASET_REPO`, as in the earlier notebooks.

In [ ]:
from google.colab import userdata

def colab_secret(name, required=True):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(
            f"Colab secret '{name}' is missing or this notebook can't read it. Open the key icon "
            "(Secrets) in the left sidebar, add it, and switch on 'Notebook access'."
        )
    return value

HF_TOKEN = colab_secret("HF_TOKEN")
HF_DATASET_REPO = HF_DATASET_REPO or colab_secret("HF_DATASET_REPO")
print("Dataset repo:", HF_DATASET_REPO)

In [ ]:
%pip install -q "huggingface_hub>=0.23.0" pyarrow

In [ ]:
import os
import subprocess
import sys

REPO_DIR = "/content/DEXTRA"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", GITHUB_REPO_URL, REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import numpy as np
import pandas as pd
import torch
from src import config
from src.data import assemble
from src.models import itransformer, sequences
from src.evaluation import metrics
from src.utils.hf_store import HFStore

itransformer.set_seed(SEED)
device = itransformer.pick_device()
store = HFStore(HF_DATASET_REPO, token=HF_TOKEN)
daily = store.read_df("processed/city_daily_features.parquet")

# Recomputed from the current config rather than trusted as saved, as in notebook 03.
saved = daily["split"].copy() if "split" in daily.columns else None
daily["split"] = assemble.assign_split(daily)
differing = int((saved != daily["split"]).sum()) if saved is not None else 0
if differing:
    print(f"! {differing:,} rows differ from the saved splits; "
          "re-run notebook 02 to bring the saved copy into line.")

print(f"device: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))
print(f"{len(daily):,} city-days, {daily.city.nunique()} cities")

## A · Windows
14 days of every variate in, three days of AQI out. Three guarantees, each with a test behind it:

- **A window never spans two cities**, so one city's last fortnight cannot become another's first.
- **Nothing after day *t* is in the input.** The window ends at *t*, the answers start at *t+1*, with no overlap of even one day.
- **A sample belongs to the split of its furthest target.** Taking the origin instead would let a window made in December 2023 put its January 2024 answers into training.

Gaps are carried forward inside a city and then filled from the pooled training median — never from a city's own future, and never from an unseen city's held-out years.

In [ ]:
X, y, index = sequences.build_sequences(daily)
variates = sequences.feature_names(daily)
target_index = variates.index(sequences.TARGET)

print(f"X {X.shape}  (windows, days, variates)")
print(f"y {y.shape}  (windows, horizons)")
display(index.split.value_counts().rename("windows").to_frame())

is_train = (index.split == "train").to_numpy()
is_val = (index.split == "val").to_numpy()
print(f"\u2713 train windows end {index.loc[is_train, 'last_target_date'].max():%Y-%m-%d}, "
      f"val windows start {index.loc[is_val, 'last_target_date'].min():%Y-%m-%d}")

## B · Train
Window normalisation is reversible and happens inside the model: each variate is standardised by its own mean and spread **within that window**, and the target's are put back on the way out. Delhi's ordinary Tuesday and Chennai's emergency stop being the same number without any global scaler, and a city the model has never seen needs no statistics of its own.

One wrinkle that is easy to get wrong: a variate that does not move inside a window would standardise to exactly zero. `archetype_id`, `coastal_flag` and `industrial_index` never move, so the three features that say *which kind of city this is* would arrive as nothing at all — and a fortnight wholly inside the stubble season would be indistinguishable from one wholly outside it. Those are passed through at their own value instead.

In [ ]:
model = itransformer.ITransformer(
    n_variates=X.shape[2], lookback=config.LOOKBACK_WINDOW,
    horizon=config.FORECAST_HORIZON, target_index=target_index,
    d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS, dropout=DROPOUT)
print(f"{itransformer.count_parameters(model):,} trainable parameters")

model, history = itransformer.train(
    model,
    itransformer.make_loader(X[is_train], y[is_train], BATCH_SIZE, shuffle=True),
    itransformer.make_loader(X[is_val], y[is_val], BATCH_SIZE * 2),
    device, epochs=EPOCHS, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    patience=PATIENCE)

### B1 · Did it learn, or did it memorise?

Expect the best epoch to arrive early — often the very first — and the curve to flatten rather than keep falling. That is not undertraining. Sweeping this architecture across a 30-fold range of capacity and a tenfold range of learning rate moved validation RMSE by 0.11 in total:

| parameters | val RMSE |
|---|---|
| 13,347 | 31.714 |
| 51,267 | 31.662 |
| 101,251 | 31.697 |
| 399,107 | 31.602 |
| 1,338,435 | 31.985 |

The ceiling is what this architecture can extract from 17,523 training windows, not a schedule that needs more patience. The only setting that helped was regularisation — dropout 0.3 and tenfold weight decay — because every configuration began overfitting within a few epochs. More capacity made it worse.

Training error here is higher than validation error, which looks wrong and is not: the training years are dirtier and more variable than 2024, so the same model scores worse on them.

In [ ]:
curve = pd.DataFrame(history).set_index("epoch")
display(curve.round(3).T)
best = curve.val_rmse.idxmin()
print(f"best epoch {best} of {len(curve)}, val rmse {curve.val_rmse.min():.3f}, "
      f"train rmse there {curve.loc[best, 'train_rmse']:.3f}")
if curve.loc[best, "train_rmse"] < 0.5 * curve.val_rmse.min():
    print("! training error is less than half the validation error - overfitting")

## C · Against the baselines
The same splits and the same metrics as notebook 03, so the numbers are directly comparable rather than merely similar.

In [ ]:
predictions = itransformer.predict(model, X, device)
# index carries last_target_date, the furthest day predicted. Each horizon lands
# on its own day, so that date is rebuilt here rather than reused.
long = pd.concat([index.assign(horizon=h + 1,
                               target_date=index["date"] + pd.Timedelta(days=h + 1),
                               y=y[:, h], itransformer=predictions[:, h])
                  for h in range(y.shape[1])], ignore_index=True)

baselines = store.read_df("processed/baseline_forecasts.parquet")
merged = long.merge(baselines[["city", "target_date", "horizon", "persistence", "climatology", "gbm"]],
                    on=["city", "target_date", "horizon"], how="inner")
print(f"{len(merged):,} forecasts shared with the baselines "
      f"(of {len(long):,} made here)")

MODELS = ["persistence", "climatology", "gbm", "itransformer"]
rows = []
for split in ["val", "test_seen", "test_unseen"]:
    part = merged[merged.split == split]
    for name in MODELS:
        row = {"split": split, "model": name}
        row.update(metrics.score(part, "y", name))
        rows.append(row)
table = pd.DataFrame(rows)[["split", "model", "n", "rmse", "mae",
                            "support", "recall", "precision", "f1"]]
display(table.round(3))

### C1 · The verdict
Beating gradient boosting on seen cities is table stakes — a tree fits these features in five seconds. The result worth having is on **unseen** cities, where notebook 03 found the booster detecting fewer hazardous days than persistence does.

In [ ]:
verdict = []
for split in ["test_seen", "test_unseen"]:
    part = merged[merged.split == split]
    itr = metrics.score(part, "y", "itransformer")
    for rival in ["persistence", "gbm"]:
        other = metrics.score(part, "y", rival)
        verdict.append({
            "split": split, "versus": rival,
            "rmse": itr["rmse"], "rmse theirs": other["rmse"],
            "rmse better by %": 100 * (1 - itr["rmse"] / other["rmse"]),
            "recall": itr["recall"], "recall theirs": other["recall"],
            "recall better by": itr["recall"] - other["recall"]})
display(pd.DataFrame(verdict).round(3))

seen = metrics.score(merged[merged.split == "test_seen"], "y", "itransformer")
unseen = metrics.score(merged[merged.split == "test_unseen"], "y", "itransformer")
print(f"transfer cost: rmse +{100 * (unseen['rmse'] / seen['rmse'] - 1):.1f}%, "
      f"recall {seen['recall']:.3f} -> {unseen['recall']:.3f} "
      f"({100 * (1 - unseen['recall'] / seen['recall']):.1f}% lost)")

### C2 · Per city and per horizon
A mean hides which cities carry it. Cities with no hazardous days show `recall` as NaN rather than 0 — nothing to find is not the same as finding nothing.

In [ ]:
per_city = []
for name in MODELS:
    part = metrics.score_by(merged[merged.split == "test_unseen"], "city", "y", name)
    part.insert(1, "model", name)
    per_city.append(part)
display(pd.concat(per_city).set_index(["city", "model"])
          [["n", "rmse", "support", "recall", "f1"]].sort_index().round(3))

by_horizon = []
for name in MODELS:
    for horizon, part in merged[merged.split == "test_seen"].groupby("horizon"):
        scored = metrics.score(part, "y", name)
        by_horizon.append({"model": name, "horizon": int(horizon),
                           "rmse": scored["rmse"], "recall": scored["recall"]})
display(pd.DataFrame(by_horizon).pivot(index="horizon", columns="model",
                                       values=["rmse", "recall"]).round(3))

## D · Save
The checkpoint and the forecasts go to Hugging Face so notebooks 05 to 08 start from exactly this model rather than retraining and getting something slightly different.

In [ ]:
import json
import tempfile
from pathlib import Path

if SAVE_TO_HF:
    spec = {"n_variates": int(X.shape[2]), "lookback": config.LOOKBACK_WINDOW,
            "horizon": config.FORECAST_HORIZON, "target_index": int(target_index),
            "d_model": D_MODEL, "n_heads": N_HEADS, "n_layers": N_LAYERS,
            "dropout": DROPOUT, "weight_decay": WEIGHT_DECAY,
            "variates": variates, "seed": SEED,
            "best_epoch": int(curve.val_rmse.idxmin()),
            "val_rmse": float(curve.val_rmse.min())}
    with tempfile.TemporaryDirectory() as tmp:
        weights = Path(tmp) / "itransformer.pt"
        torch.save(model.state_dict(), weights)
        store.upload_file(weights, "models/itransformer.pt", "iTransformer backbone weights")
    store.upload_json(spec, "models/itransformer.json", "iTransformer architecture and variate order")
    keep = ["city", "date", "target_date", "horizon", "split", "y", "itransformer"]
    store.upload_df(long[keep], "processed/itransformer_forecasts.parquet",
                    "iTransformer forecasts")
    store.upload_df(pd.DataFrame(history), "processed/itransformer_history.csv",
                    "iTransformer training curve")
    print("\u2191 models/itransformer.pt, models/itransformer.json")
    print("\u2191 processed/itransformer_forecasts.parquet, itransformer_history.csv")
else:
    print("SAVE_TO_HF is off; nothing uploaded.")

print("\nNext: notebook 05 adds the meteorological gate and the dual experts.")